# Act 5 — The outlier plot

> **Opening question:** *"Some houses in this dataset sell for a fraction of what they should be worth — and others sell for far more. Are those just data errors? Or are they the most interesting properties in the whole city?"*

---
Outliers are the moments where the story breaks from the script.
This act finds them, names them, and decides: mistake or masterpiece?

In [ ]:
import sys
sys.path.append('..')
from src.utils import *

act_header(
    act_num=5,
    title="The outlier plot",
    opening_question="Which houses break the price-size rule — and is that because of bad data or genuinely extraordinary properties?"
)

df = load_processed('cleaned.csv')
df = df.reset_index(drop=True)
df['HouseID'] = df.index + 1

## 5.1 — Flag price outliers using IQR

In [ ]:
df = flag_outliers_iqr(df, 'SalePrice')
df = flag_outliers_iqr(df, 'GrLivArea')

n_price_out = df['SalePrice_is_outlier'].sum()
n_area_out  = df['GrLivArea_is_outlier'].sum()
n_both      = (df['SalePrice_is_outlier'] & df['GrLivArea_is_outlier']).sum()

print(f"\n  Price outliers:      {n_price_out} houses")
print(f"  Area outliers:       {n_area_out} houses")
print(f"  Outliers on BOTH:    {n_both} houses — these are the most unusual")

## 5.2 — The outlier scatter: size vs price, flagged
Hover over any dot to see the house details.

In [ ]:
df['_outlier_label'] = 'Normal'
df.loc[df['SalePrice_is_outlier'] & ~df['GrLivArea_is_outlier'],  '_outlier_label'] = 'Price outlier'
df.loc[~df['SalePrice_is_outlier'] & df['GrLivArea_is_outlier'],  '_outlier_label'] = 'Size outlier'
df.loc[df['SalePrice_is_outlier'] & df['GrLivArea_is_outlier'],   '_outlier_label'] = 'Both outlier'

color_map = {
    'Normal':        '#534AB7',
    'Price outlier': '#D85A30',
    'Size outlier':  '#BA7517',
    'Both outlier':  '#D4537E'
}

hover_cols = ['HouseID', 'OverallQual', 'YearBuilt']
if 'Neighborhood' in df.columns:
    hover_cols.append('Neighborhood')
if 'SaleCondition' in df.columns:
    hover_cols.append('SaleCondition')

fig = px.scatter(
    df, x='GrLivArea', y='SalePrice',
    color='_outlier_label',
    color_discrete_map=color_map,
    hover_data=hover_cols,
    title='Size vs price — finding the houses that break the pattern',
    labels={
        'GrLivArea': 'Above-ground living area (sq ft)',
        'SalePrice': 'Sale price ($)',
        '_outlier_label': 'Category'
    },
    template='plotly_white',
    opacity=0.7
)
fig.update_traces(marker=dict(size=7))
fig.update_layout(
    title_font_size=14,
    yaxis_tickprefix='$', yaxis_tickformat=',.0f',
    margin=dict(t=60, b=40),
    legend_title=''
)
save_plotly(fig, 'act5_outlier_scatter.html')
fig.show()

## 5.3 — The most suspicious outliers
Large houses selling at low prices — are these data errors or distressed sales?

In [ ]:
# Homes with huge area but very low price — the most suspicious
suspicious = df[
    (df['GrLivArea'] > 4000) & (df['SalePrice'] < df['SalePrice'].quantile(0.5))
].copy()

display_cols = ['HouseID', 'GrLivArea', 'SalePrice', 'OverallQual', 'YearBuilt']
if 'SaleCondition' in df.columns:
    display_cols.append('SaleCondition')
if 'Neighborhood' in df.columns:
    display_cols.append('Neighborhood')

existing_cols = [c for c in display_cols if c in suspicious.columns]

if len(suspicious) > 0:
    print(f"\n  Found {len(suspicious)} house(s) with >4,000 sq ft but below-median price:\n")
    print(suspicious[existing_cols].to_string(index=False))
    insight_callout(
        "These large, cheap houses are the most important outliers in the dataset.\n"
        "Kaggle competition notes confirm 2 of these are data entry errors — they should\n"
        "be removed before building a price prediction model. The others may be\n"
        "genuine distressed or partial sales (check SaleCondition = 'Partial').",
        label="Data error or real?"
    )
else:
    print("  No highly suspicious outliers found with current thresholds.")

## 5.4 — Price outliers by neighbourhood
Some neighbourhoods consistently produce the high-end outliers.

In [ ]:
if 'Neighborhood' in df.columns:
    price_outliers = df[df['SalePrice_is_outlier']]
    nbr_outlier_counts = price_outliers['Neighborhood'].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(nbr_outlier_counts.index[::-1], nbr_outlier_counts.values[::-1],
            color=PALETTE['coral'], alpha=0.85)
    ax.set_xlabel('Number of price outlier sales')
    ax.set_title('Which neighbourhoods produce the most price outliers?', fontsize=13)
    plt.tight_layout()
    save_figure(fig, 'act5_neighborhood_outliers.png')
    plt.show()

    insight_callout(
        f"'{nbr_outlier_counts.index[0]}' has the most price outlier sales.\n"
        "This doesn't mean it's a bad neighbourhood — it means it has the highest\n"
        "variance: some exceptional homes sit alongside more modest ones."
    )

---
## Act 5 — Closing punchline

In [ ]:
punchline(
    "Most outliers in Ames are genuine: luxury homes, partial sales, or unusual lot configurations. "
    "Two specific large-area, low-price entries appear to be data errors and should be removed before modelling. "
    "In Act 6, we bring every insight together into one final answer."
)

**Next → [Act 6 — The Punchline](06_act6_punchline.ipynb)**